# Inertia Tensors for Primitive Shapes

**Primary Reference:** Meriam & Kraige, *Engineering Mechanics: Dynamics*, Appendix B  
**Supporting References:**  
- Beer, Johnston & Cornwell, *Vector Mechanics for Engineers: Dynamics*, Ch. 9  
- MIT OCW 2.003 Engineering Dynamics — https://ocw.mit.edu/courses/2-003sc-engineering-dynamics-fall-2011  

---

## The Inertia Tensor

The inertia tensor $\mathcal{I}$ is a 3x3 symmetric matrix that describes how mass is distributed around each axis:

$$
\mathcal{I} = \begin{bmatrix} I_{xx} & -I_{xy} & -I_{xz} \\ -I_{xy} & I_{yy} & -I_{yz} \\ -I_{xz} & -I_{yz} & I_{zz} \end{bmatrix}
$$

For primitive shapes aligned with their symmetry axis, the off-diagonal terms ($I_{xy}$, $I_{xz}$, $I_{yz}$) are zero, reducing $\mathcal{I}$ to a diagonal matrix. This project assumes shapes are aligned with their joint frame, so all off-diagonal terms are zero.

---

## Solid Cylinder

Mass $m$, radius $r$, length $L$. Symmetry axis along $z$.

$$
I_{xx} = I_{yy} = \frac{1}{12} m (3r^2 + L^2)
$$
$$
I_{zz} = \frac{1}{2} m r^2
$$

---

## Hollow Cylinder (Tube)

Mass $m$, outer radius $r_o$, inner radius $r_i$, length $L$. Symmetry axis along $z$.

$$
I_{xx} = I_{yy} = \frac{1}{12} m (3(r_o^2 + r_i^2) + L^2)
$$
$$
I_{zz} = \frac{1}{2} m (r_o^2 + r_i^2)
$$

---

## Rectangular Box

Mass $m$, width $w$ (along $x$), height $h$ (along $y$), length $L$ (along $z$).

$$
I_{xx} = \frac{1}{12} m (h^2 + L^2)
$$
$$
I_{yy} = \frac{1}{12} m (w^2 + L^2)
$$
$$
I_{zz} = \frac{1}{12} m (w^2 + h^2)
$$

---

## Parallel Axis Theorem

The formulas above give $\mathcal{I}$ about the **center of mass**. If you need $\mathcal{I}$ about a different point (e.g. the joint origin), use the parallel axis theorem:

$$
I_{xx}' = I_{xx} + m(d_y^2 + d_z^2)
$$

Where $d_y$ and $d_z$ are the distances from the center of mass to the new axis along $y$ and $z$ respectively. The same applies for $I_{yy}'$ and $I_{zz}'$.

In matrix form:

$$
\mathcal{I}' = \mathcal{I}_{cm} + m \left( (\mathbf{d}^T \mathbf{d}) \mathbf{1} - \mathbf{d} \mathbf{d}^T \right)
$$

Where $\mathbf{d}$ is the vector from the center of mass to the new reference point.

For this project, $\mathcal{I}$ is computed about the center of mass and the Lagrangian formulation uses it there directly — so the parallel axis theorem is not needed unless you later want inertia about the joint origin.

**Reference:** Beer & Johnston Ch. 9.6

---

## What to Implement in `inertia_tensor.py`

One function per geometry type, each returning a 3x3 sympy diagonal Matrix:

- `inertia_cylinder(m, r, L)` — solid cylinder
- `inertia_hollow_cylinder(m, r_outer, r_inner, L)` — hollow cylinder / tube
- `inertia_box(m, w, h, L)` — rectangular box

A dispatcher function reads `joint["geometry"]` from the YAML and calls the appropriate function:

- `get_inertia(joint)` — reads geometry type and parameters, returns $\mathcal{I}_i$

## Example — Solid Cylinder

In [ ]:
import sympy as smp

def inertia_cylinder(m, r, L):
    Ixx = smp.Rational(1, 12) * m * (3*r**2 + L**2)
    Izz = smp.Rational(1, 2) * m * r**2
    return smp.diag(Ixx, Ixx, Izz)  # symmetry axis along z

# smp.Rational preserves exact fractions — smp.Rational(1,12) stays as 1/12
# rather than being converted to the float 0.08333...
# smp.diag builds a diagonal matrix from the given values